# Reranker Fine-tuning — Ray Train
Run cells top to bottom. GPU runtime required.

In [ ]:
# Cell 1 — verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Cell 2 — install deps
!pip install ray[train] transformers torch qdrant-client sentence-transformers -q

In [ ]:
# Cell 3 — clone repo
!git clone https://github.com/fbarulli/RAG-a-muffin.git
%cd RAG-a-muffin

In [ ]:
# Cell 4 — start Cloudflare tunnel (run on your VM before this cell)
# cloudflared tunnel --url http://localhost:6333
# paste the resulting URL below
QDRANT_URL = "https://began-verbal-load-low.trycloudflare.com"  # replace each session

from qdrant_client import QdrantClient
client = QdrantClient(url=QDRANT_URL, prefer_grpc=False, https=True, port=443)
print(client.get_collections())

In [ ]:
# Cell 5 — verify triples exist
import json, pathlib
triples_path = pathlib.Path('experiments/reranker_training/triples_sample_200.json')
triples = json.loads(triples_path.read_text())
print(f'Triples loaded: {len(triples)}')
from collections import Counter
print('Distribution:', dict(Counter(t.get('course', 'unknown') for t in triples)))

In [ ]:
# Cell 6 — verify config
import json, pathlib
cfg = json.loads(pathlib.Path('configs/rerankers.json').read_text())
rt = cfg['ray_training']
print('model_key:        ', rt['model_key'])
print('sample_size:      ', cfg['training']['sample_size'])
print('use_gpu:          ', rt['use_gpu'])
print('fp16:             ', rt['fp16'])
print('epochs:           ', rt['epochs'])
print('batch_size:       ', rt['batch_size'])
print('log_every_n_steps:', rt['log_every_n_steps'])

In [ ]:
# Cell 7 — set use_gpu + fp16 True for Colab GPU
import json, pathlib
p = pathlib.Path('configs/rerankers.json')
cfg = json.loads(p.read_text())
cfg['ray_training']['use_gpu'] = True
cfg['ray_training']['fp16'] = True
p.write_text(json.dumps(cfg, indent=2))
print('use_gpu:', cfg['ray_training']['use_gpu'])
print('fp16:   ', cfg['ray_training']['fp16'])

In [ ]:
# Cell 8 — run training
import sys
sys.path.insert(0, 'src')

from rag_pipeline.ingestion.reranking.reranking_training_ray import main
main()

In [ ]:
# Cell 9 — verify output
import os
out = 'experiments/reranker_models/TinyBERT-finetuned-test'
print(os.listdir(out) if os.path.exists(out) else 'NOT FOUND')